Khởi tạo và tải dữ liệu

In [ ]:
import pandas as pd
import numpy as np
import re
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, SpatialDropout1D
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score

# Thông báo bắt đầu
print("[THÔNG BÁO] Đang khởi tạo môi trường làm việc...")
print("[THÔNG BÁO] Đang tải tập dữ liệu từ file 'mbti_1.csv'...")

try:
    df = pd.read_csv('mbti_1.csv')
    print(f"[THÀNH CÔNG] Tải dữ liệu hoàn tất. Tổng số bản ghi: {len(df)}")
except FileNotFoundError:
    print("[LỖI] Không tìm thấy file 'mbti_1.csv'. Vui lòng kiểm tra lại file đã tải lên Colab chưa.")
except Exception as e:
    print(f"[LỖI] Đã xảy ra lỗi không mong muốn: {e}")

[THÔNG BÁO] Đang khởi tạo môi trường làm việc...
[THÔNG BÁO] Đang tải tập dữ liệu từ file 'mbti_1.csv'...
[THÀNH CÔNG] Tải dữ liệu hoàn tất. Tổng số bản ghi: 8675


Tiền xử lý dữ liệu

In [ ]:
# 1. Hàm làm sạch văn bản
def clean_text(text):
    """
    Hàm xử lý văn bản:
    - Chuyển đổi về chữ thường.
    - Loại bỏ URL và liên kết.
    - Loại bỏ các từ khóa MBTI để ngăn chặn rò rỉ dữ liệu (Data Leakage).
    - Loại bỏ các ký tự đặc biệt.
    """
    text = str(text).lower()
    text = re.sub(r'http\S+', '', text) # Loại bỏ URL

    # Danh sách 16 từ khóa MBTI cần loại bỏ
    mbti_types = ['infj', 'entp', 'intp', 'intj', 'entj', 'enfj', 'infp', 'enfp',
                  'isfp', 'istp', 'isfj', 'istj', 'estp', 'esfp', 'estj', 'esfj']
    pattern = r'\b(?:' + '|'.join(mbti_types) + r')\b'
    text = re.sub(pattern, '', text)

    text = re.sub(r'[^a-z\s]', ' ', text) # Chỉ giữ lại chữ cái
    text = re.sub(r'\s+', ' ', text).strip() # Chuẩn hóa khoảng trắng
    return text

print("[THÔNG BÁO] Đang làm sạch dữ liệu và loại bỏ từ khóa MBTI...")
df['clean_posts'] = df['posts'].apply(clean_text)

# 2. Mã hóa nhãn thành 4 trục nhị phân
print("[THÔNG BÁO] Đang mã hóa nhãn thành 4 trục nhị phân (IE, NS, TF, JP)...")

def get_binary_labels(mbti_type):
    return pd.Series({
        'IE': 1 if 'I' in mbti_type else 0, # Hướng nội (1) vs Hướng ngoại (0)
        'NS': 1 if 'N' in mbti_type else 0, # Trực giác (1) vs Giác quan (0)
        'TF': 1 if 'T' in mbti_type else 0, # Lý trí (1) vs Cảm xúc (0)
        'JP': 1 if 'J' in mbti_type else 0  # Nguyên tắc (1) vs Linh hoạt (0)
    })

binary_targets = df['type'].apply(get_binary_labels)
df = pd.concat([df, binary_targets], axis=1)

print("[THÀNH CÔNG] Tiền xử lý dữ liệu hoàn tất.")

[THÔNG BÁO] Đang làm sạch dữ liệu và loại bỏ từ khóa MBTI...
[THÔNG BÁO] Đang mã hóa nhãn thành 4 trục nhị phân (IE, NS, TF, JP)...
[THÀNH CÔNG] Tiền xử lý dữ liệu hoàn tất.


Word Embedding: Mã hóa văn bản & padding

In [ ]:
# Cấu hình siêu tham số
MAX_NB_WORDS = 20000       # Số lượng từ vựng tối đa
MAX_SEQUENCE_LENGTH = 500  # Độ dài cố định của chuỗi đầu vào

print("[THÔNG BÁO] Đang chia tập dữ liệu Huấn luyện (Train) và Kiểm thử (Test) với tỷ lệ 80/20...")
X_train_text, X_test_text, y_train_multi, y_test_multi = train_test_split(
    df['clean_posts'],
    df[['IE', 'NS', 'TF', 'JP']],
    test_size=0.2,
    random_state=42,
    stratify=df['type']
)

print("[THÔNG BÁO] Đang huấn luyện bộ Tokenizer trên tập dữ liệu huấn luyện...")
tokenizer = Tokenizer(num_words=MAX_NB_WORDS, oov_token='<OOV>')
tokenizer.fit_on_texts(X_train_text)

print("[THÔNG BÁO] Đang chuyển đổi văn bản sang chuỗi số và thực hiện Padding...")
X_train_seq = tokenizer.texts_to_sequences(X_train_text)
X_test_seq = tokenizer.texts_to_sequences(X_test_text)

X_train_pad = pad_sequences(X_train_seq, maxlen=MAX_SEQUENCE_LENGTH)
X_test_pad = pad_sequences(X_test_seq, maxlen=MAX_SEQUENCE_LENGTH)

print(f"[THÀNH CÔNG] Mã hóa hoàn tất. Kích thước tập huấn luyện: {X_train_pad.shape}")

[THÔNG BÁO] Đang chia tập dữ liệu Huấn luyện (Train) và Kiểm thử (Test) với tỷ lệ 80/20...
[THÔNG BÁO] Đang huấn luyện bộ Tokenizer trên tập dữ liệu huấn luyện...
[THÔNG BÁO] Đang chuyển đổi văn bản sang chuỗi số và thực hiện Padding...
[THÀNH CÔNG] Mã hóa hoàn tất. Kích thước tập huấn luyện: (6940, 500)


Huấn luyện mô hình LSTM

In [ ]:
# Cấu hình tham số mô hình
EMBEDDING_DIM = 100
EPOCHS = 20
BATCH_SIZE = 64

def build_lstm_model():
    """
    Xây dựng mô hình LSTM với lớp Embedding tự học.
    """
    model = Sequential()
    model.add(Embedding(MAX_NB_WORDS, EMBEDDING_DIM, input_length=MAX_SEQUENCE_LENGTH))
    model.add(SpatialDropout1D(0.2))
    model.add(LSTM(64, dropout=0.2, recurrent_dropout=0.0))
    model.add(Dense(1, activation='sigmoid'))
    model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
    return model

target_cols = ['IE', 'NS', 'TF', 'JP']
trained_models = {} # Từ điển để lưu trữ 4 mô hình sau khi huấn luyện

print("="*60)
print("[THÔNG BÁO] BẮT ĐẦU QUÁ TRÌNH HUẤN LUYỆN 4 MÔ HÌNH NHỊ PHÂN")
print("="*60)

for col in target_cols:
    print(f"\n[THÔNG BÁO] Đang khởi tạo và huấn luyện mô hình cho trục: {col}...")

    y_train = y_train_multi[col].values

    # Khởi tạo mô hình mới
    model = build_lstm_model()

    # Cấu hình Dừng sớm (Early Stopping)
    early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

    # Huấn luyện
    model.fit(X_train_pad, y_train, epochs=EPOCHS, batch_size=BATCH_SIZE,
              validation_split=0.1, callbacks=[early_stop], verbose=1)

    # Lưu mô hình vào từ điển để sử dụng ở bước sau
    trained_models[col] = model
    print(f"[THÀNH CÔNG] Đã huấn luyện và lưu mô hình trục {col}.")

print("\n[THÔNG BÁO] QUÁ TRÌNH HUẤN LUYỆN HOÀN TẤT. CHUYỂN SANG BƯỚC ĐÁNH GIÁ.")

[THÔNG BÁO] BẮT ĐẦU QUÁ TRÌNH HUẤN LUYỆN 4 MÔ HÌNH NHỊ PHÂN

[THÔNG BÁO] Đang khởi tạo và huấn luyện mô hình cho trục: IE...
Epoch 1/20


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


98/98 ━━━━━━━━━━━━━━━━━━━━ 4s 25ms/step - accuracy: 0.7688 - loss: 0.5807 - val_accuracy: 0.7536 - val_loss: 0.5610
Epoch 2/20
98/98 ━━━━━━━━━━━━━━━━━━━━ 2s 21ms/step - accuracy: 0.7741 - loss: 0.5135 - val_accuracy: 0.7536 - val_loss: 0.5767
Epoch 3/20
98/98 ━━━━━━━━━━━━━━━━━━━━ 2s 21ms/step - accuracy: 0.8068 - loss: 0.4181 - val_accuracy: 0.7046 - val_loss: 0.6314
Epoch 4/20
98/98 ━━━━━━━━━━━━━━━━━━━━ 2s 21ms/step - accuracy: 0.8815 - loss: 0.2934 - val_accuracy: 0.6210 - val_loss: 0.7661
[THÀNH CÔNG] Đã huấn luyện và lưu mô hình trục IE.

[THÔNG BÁO] Đang khởi tạo và huấn luyện mô hình cho trục: NS...
Epoch 1/20
98/98 ━━━━━━━━━━━━━━━━━━━━ 5s 25ms/step - accuracy: 0.8419 - loss: 0.4940 - val_accuracy: 0.8617 - val_loss: 0.4021
Epoch 2/20
98/98 ━━━━━━━━━━━━━━━━━━━━ 2s 21ms/step - accuracy: 0.8703 - loss: 0.3785 - val_accuracy: 0.8617 - val_loss: 0.4001
Epoch 3/20
98/98 ━━━━━━━━━━━━━━━━━━━━ 2s 21ms/step - accuracy: 0.8636 - loss: 0.3313 - val_accuracy: 0.8588 - val_loss: 0.4247
Epoch 

Báo cáo Kết quả

In [ ]:
print("="*60)
print("[THÔNG BÁO] BẮT ĐẦU QUÁ TRÌNH ĐÁNH GIÁ TRÊN TẬP KIỂM THỬ (TEST SET)")
print("="*60)

# Dictionary để tổng hợp độ chính xác cuối cùng
final_results = {}

for col in target_cols:
    print(f"\n[THÔNG BÁO] Đang đánh giá mô hình trục: {col}...")

    # Truy xuất mô hình tương ứng từ bộ nhớ
    if col not in trained_models:
        print(f"[LỖI] Không tìm thấy mô hình cho trục {col}. Vui lòng chạy lại Ô 4.")
        continue

    model = trained_models[col]
    y_test = y_test_multi[col].values

    # Thực hiện dự đoán
    y_pred_prob = model.predict(X_test_pad, verbose=0)
    y_pred = (y_pred_prob > 0.5).astype(int)

    # Tính toán các chỉ số
    acc = accuracy_score(y_test, y_pred)
    final_results[col] = acc

    # In kết quả
    print(f"[KẾT QUẢ] Độ chính xác (Accuracy) cho trục {col}: {acc:.2%}")
    print("-" * 30)
    print("Báo cáo phân loại chi tiết (Classification Report):")
    print(classification_report(y_test, y_pred, digits=4))

print("\n" + "="*60)
print("TỔNG HỢP KẾT QUẢ CUỐI CÙNG (WORD EMBEDDING + LSTM)")
print("="*60)
for col, acc in final_results.items():
    print(f" - Trục {col}: {acc:.2%}")
print("="*60)

[THÔNG BÁO] BẮT ĐẦU QUÁ TRÌNH ĐÁNH GIÁ TRÊN TẬP KIỂM THỬ (TEST SET)

[THÔNG BÁO] Đang đánh giá mô hình trục: IE...
[KẾT QUẢ] Độ chính xác (Accuracy) cho trục IE: 76.83%
------------------------------
Báo cáo phân loại chi tiết (Classification Report):
              precision    recall  f1-score   support

           0     0.0000    0.0000    0.0000       401
           1     0.7687    0.9993    0.8690      1334

    accuracy                         0.7683      1735
   macro avg     0.3844    0.4996    0.4345      1735
weighted avg     0.5911    0.7683    0.6681      1735


[THÔNG BÁO] Đang đánh giá mô hình trục: NS...
[KẾT QUẢ] Độ chính xác (Accuracy) cho trục NS: 86.17%
------------------------------
Báo cáo phân loại chi tiết (Classification Report):
              precision    recall  f1-score   support

           0     0.0000    0.0000    0.0000       240
           1     0.8617    1.0000    0.9257      1495

    accuracy                         0.8617      1735
   macro avg     0.

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


[KẾT QUẢ] Độ chính xác (Accuracy) cho trục TF: 55.04%
------------------------------
Báo cáo phân loại chi tiết (Classification Report):
              precision    recall  f1-score   support

           0     0.5626    0.7604    0.6467       939
           1     0.5172    0.3028    0.3819       796

    accuracy                         0.5504      1735
   macro avg     0.5399    0.5316    0.5143      1735
weighted avg     0.5418    0.5504    0.5252      1735


[THÔNG BÁO] Đang đánh giá mô hình trục: JP...
[KẾT QUẢ] Độ chính xác (Accuracy) cho trục JP: 60.40%
------------------------------
Báo cáo phân loại chi tiết (Classification Report):
              precision    recall  f1-score   support

           0     0.6040    1.0000    0.7531      1048
           1     0.0000    0.0000    0.0000       687

    accuracy                         0.6040      1735
   macro avg     0.3020    0.5000    0.3766      1735
weighted avg     0.3649    0.6040    0.4549      1735


TỔNG HỢP KẾT QUẢ CUỐI CÙ

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


DO MÔ HÌNH TRÊN BỊ NGÔ NÊN TRAIN LẠI >< (file báo cáo có nói vì sao nó bị ngô)

B1: Khởi tạo Môi trường và Tải Tài nguyên

In [ ]:
import pandas as pd
import numpy as np
import re
import os
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, SpatialDropout1D, Bidirectional
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
from sklearn.utils.class_weight import compute_class_weight

# 1. Tải bộ GloVe Embeddings (Phiên bản 6B, vector 100 chiều)
print("[THÔNG BÁO] Đang tải bộ dữ liệu GloVe Embeddings từ máy chủ Stanford...")
# Sử dụng lệnh hệ thống để tải file zip
!wget http://nlp.stanford.edu/data/glove.6B.zip

print("[THÔNG BÁO] Đang giải nén tập tin glove.6B.zip...")
!unzip -q glove.6B.zip

# 2. Tải dữ liệu MBTI
print("[THÔNG BÁO] Đang tải dữ liệu từ tập tin 'mbti_1.csv'...")
try:
    df = pd.read_csv('mbti_1.csv', on_bad_lines='skip', engine='python')
    print(f"[THÀNH CÔNG] Tải dữ liệu hoàn tất. Tổng số bản ghi: {len(df)}")
except FileNotFoundError:
    print("[LỖI] Không tìm thấy tập tin 'mbti_1.csv'. Vui lòng kiểm tra lại việc tải lên.")
except Exception as e:
    print(f"[LỖI] Đã xảy ra lỗi ngoại lệ: {e}")

[THÔNG BÁO] Đang tải bộ dữ liệu GloVe Embeddings từ máy chủ Stanford...
--2025-12-12 14:32:36--  http://nlp.stanford.edu/data/glove.6B.zip
Resolving nlp.stanford.edu (nlp.stanford.edu)... 171.64.67.140
Connecting to nlp.stanford.edu (nlp.stanford.edu)|171.64.67.140|:80... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://nlp.stanford.edu/data/glove.6B.zip [following]
--2025-12-12 14:32:36--  https://nlp.stanford.edu/data/glove.6B.zip
Connecting to nlp.stanford.edu (nlp.stanford.edu)|171.64.67.140|:443... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://downloads.cs.stanford.edu/nlp/data/glove.6B.zip [following]
--2025-12-12 14:32:37--  https://downloads.cs.stanford.edu/nlp/data/glove.6B.zip
Resolving downloads.cs.stanford.edu (downloads.cs.stanford.edu)... 171.64.64.22
Connecting to downloads.cs.stanford.edu (downloads.cs.stanford.edu)|171.64.64.22|:443... connected.
HTTP request sent, awaiting response... 200 OK
Le

B2: Tiền xử lý và Xây dựng Ma trận Embedding

In [ ]:
# Cấu hình siêu tham số
MAX_NB_WORDS = 20000       # Giới hạn số lượng từ vựng
MAX_SEQUENCE_LENGTH = 500  # Độ dài chuỗi đầu vào
EMBEDDING_DIM = 100        # Số chiều của vector GloVe

# 1. Hàm làm sạch văn bản
def clean_text_formal(text):
    """
    Hàm xử lý văn bản chuẩn hóa:
    - Chuyển đổi về chữ thường.
    - Loại bỏ URL và các ký tự đặc biệt.
    - Loại bỏ các từ khóa MBTI để ngăn chặn rò rỉ dữ liệu (Data Leakage).
    """
    text = str(text).lower()
    text = re.sub(r'http\S+', '', text)

    # Danh sách từ khóa MBTI cần loại bỏ
    mbti_types = ['infj', 'entp', 'intp', 'intj', 'entj', 'enfj', 'infp', 'enfp',
                  'isfp', 'istp', 'isfj', 'istj', 'estp', 'esfp', 'estj', 'esfj']
    pattern = r'\b(?:' + '|'.join(mbti_types) + r')\b'
    text = re.sub(pattern, '', text)

    text = re.sub(r'[^a-z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

print("[THÔNG BÁO] Đang thực hiện làm sạch dữ liệu...")
df['clean_posts'] = df['posts'].apply(clean_text_formal)

# 2. Mã hóa nhãn dữ liệu (Binary Encoding)
def get_binary_labels(mbti_type):
    return pd.Series({
        'IE': 1 if 'I' in mbti_type else 0,
        'NS': 1 if 'N' in mbti_type else 0,
        'TF': 1 if 'T' in mbti_type else 0,
        'JP': 1 if 'J' in mbti_type else 0
    })
binary_targets = df['type'].apply(get_binary_labels)
df = pd.concat([df, binary_targets], axis=1)

# 3. Phân chia tập dữ liệu và Tokenization
print("[THÔNG BÁO] Đang phân chia tập Huấn luyện/Kiểm thử và thực hiện Tokenization...")
X_train_text, X_test_text, y_train_multi, y_test_multi = train_test_split(
    df['clean_posts'], df[['IE', 'NS', 'TF', 'JP']], test_size=0.2, random_state=42, stratify=df['type']
)

tokenizer = Tokenizer(num_words=MAX_NB_WORDS, oov_token='<OOV>')
tokenizer.fit_on_texts(X_train_text)

X_train_seq = tokenizer.texts_to_sequences(X_train_text)
X_test_seq = tokenizer.texts_to_sequences(X_test_text)

X_train_pad = pad_sequences(X_train_seq, maxlen=MAX_SEQUENCE_LENGTH)
X_test_pad = pad_sequences(X_test_seq, maxlen=MAX_SEQUENCE_LENGTH)

# 4. Tích hợp GloVe Embeddings
print("[THÔNG BÁO] Đang nạp từ điển vector GloVe vào bộ nhớ...")
embeddings_index = {}
try:
    f = open('glove.6B.100d.txt', encoding='utf-8')
    for line in f:
        values = line.split()
        word = values[0]
        coefs = np.asarray(values[1:], dtype='float32')
        embeddings_index[word] = coefs
    f.close()
    print(f"[THÔNG BÁO] Đã nạp thành công {len(embeddings_index)} vector từ GloVe.")
except FileNotFoundError:
    print("[LỖI] Không tìm thấy file GloVe. Vui lòng kiểm tra lại bước tải xuống.")

# Tạo ma trận trọng số Embedding
print("[THÔNG BÁO] Đang xây dựng Ma trận Embedding (Embedding Matrix)...")
word_index = tokenizer.word_index
embedding_matrix = np.zeros((MAX_NB_WORDS, EMBEDDING_DIM))
hits = 0
misses = 0

for word, i in word_index.items():
    if i < MAX_NB_WORDS:
        embedding_vector = embeddings_index.get(word)
        if embedding_vector is not None:
            embedding_matrix[i] = embedding_vector
            hits += 1
        else:
            misses += 1

print(f"[THỐNG KÊ] Số từ vựng khớp với GloVe: {hits}")
print(f"[THỐNG KÊ] Số từ vựng mới (không có trong GloVe - sẽ được học thêm): {misses}")

[THÔNG BÁO] Đang thực hiện làm sạch dữ liệu...
[THÔNG BÁO] Đang phân chia tập Huấn luyện/Kiểm thử và thực hiện Tokenization...
[THÔNG BÁO] Đang nạp từ điển vector GloVe vào bộ nhớ...
[THÔNG BÁO] Đã nạp thành công 400000 vector từ GloVe.
[THÔNG BÁO] Đang xây dựng Ma trận Embedding (Embedding Matrix)...
[THỐNG KÊ] Số từ vựng khớp với GloVe: 19190
[THỐNG KÊ] Số từ vựng mới (không có trong GloVe - sẽ được học thêm): 809


B3: Hàm tính toán Trọng số Lớp (Class Weights Utility)

In [ ]:
def calculate_class_weights(y_train_data):
    """
    Tính toán trọng số lớp để xử lý mất cân bằng dữ liệu.
    - Công thức: n_samples / (n_classes * n_samples_j)
    - Mục đích: Tăng trọng số phạt cho các lớp thiểu số trong hàm mất mát.
    """
    weights = compute_class_weight(
        class_weight='balanced',
        classes=np.unique(y_train_data),
        y=y_train_data
    )
    # Chuyển đổi sang định dạng từ điển {class_label: weight}
    return dict(enumerate(weights))

B4: Huấn luyện Mô hình

In [ ]:
EPOCHS = 20
BATCH_SIZE = 64

def build_fine_tuned_lstm_model():
    """
    Xây dựng mô hình Bi-Directional LSTM với GloVe Embeddings.
    Lớp Embedding được thiết lập trainable=True để cho phép tinh chỉnh (fine-tuning).
    """
    model = Sequential()

    # Lớp Embedding khởi tạo bằng trọng số GloVe
    model.add(Embedding(MAX_NB_WORDS, EMBEDDING_DIM,
                        input_length=MAX_SEQUENCE_LENGTH,
                        weights=[embedding_matrix],
                        trainable=True)) # Cho phép học các từ vựng mới/từ lóng

    model.add(SpatialDropout1D(0.2))

    # Sử dụng LSTM hai chiều (Bidirectional) để nắm bắt ngữ cảnh tốt hơn
    model.add(Bidirectional(LSTM(64, dropout=0.2, recurrent_dropout=0.0)))

    model.add(Dense(1, activation='sigmoid'))
    model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
    return model

target_cols = ['IE', 'NS', 'TF', 'JP']

print("="*80)
print("BẮT ĐẦU QUÁ TRÌNH HUẤN LUYỆN: GLOVE EMBEDDINGS + BI-LSTM + CLASS WEIGHTS")
print("="*80)

for col in target_cols:
    print(f"\n[THÔNG BÁO] Đang huấn luyện mô hình cho trục: {col}...")

    y_train = y_train_multi[col].values
    y_test = y_test_multi[col].values

    # 1. Tính toán trọng số lớp cho trục hiện tại
    cls_weights = calculate_class_weights(y_train)
    print(f"   -> Trọng số lớp áp dụng (Class Weights): {cls_weights}")

    # 2. Khởi tạo và Huấn luyện mô hình
    model = build_fine_tuned_lstm_model()

    # Cấu hình Dừng sớm
    early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

    # Huấn luyện với class_weight
    model.fit(X_train_pad, y_train,
              epochs=EPOCHS, batch_size=BATCH_SIZE,
              validation_split=0.1,
              callbacks=[early_stop],
              class_weight=cls_weights,
              verbose=1)

    # 3. Đánh giá kết quả
    y_pred_prob = model.predict(X_test_pad)
    y_pred = (y_pred_prob > 0.5).astype(int)

    acc = accuracy_score(y_test, y_pred)
    print(f"[KẾT QUẢ] Độ chính xác (Accuracy) cho trục {col}: {acc:.2%}")
    print("Báo cáo phân loại chi tiết (Classification Report):")
    print(classification_report(y_test, y_pred, digits=4))

print("\n[THÔNG BÁO] QUÁ TRÌNH THỰC NGHIỆM GIAI ĐOẠN 2 ĐÃ HOÀN TẤT.")

BẮT ĐẦU QUÁ TRÌNH HUẤN LUYỆN: GLOVE EMBEDDINGS + BI-LSTM + CLASS WEIGHTS

[THÔNG BÁO] Đang huấn luyện mô hình cho trục: IE...
   -> Trọng số lớp áp dụng (Class Weights): {0: np.float64(2.1714643304130163), 1: np.float64(0.649569449644328)}


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Epoch 1/20
98/98 ━━━━━━━━━━━━━━━━━━━━ 11s 43ms/step - accuracy: 0.4845 - loss: 0.7015 - val_accuracy: 0.4914 - val_loss: 0.6963
Epoch 2/20
98/98 ━━━━━━━━━━━━━━━━━━━━ 4s 38ms/step - accuracy: 0.5355 - loss: 0.6905 - val_accuracy: 0.5173 - val_loss: 0.6930
Epoch 3/20
98/98 ━━━━━━━━━━━━━━━━━━━━ 5s 50ms/step - accuracy: 0.5884 - loss: 0.6748 - val_accuracy: 0.4885 - val_loss: 0.7013
Epoch 4/20
98/98 ━━━━━━━━━━━━━━━━━━━━ 4s 38ms/step - accuracy: 0.6215 - loss: 0.6601 - val_accuracy: 0.4236 - val_loss: 0.7405
Epoch 5/20
98/98 ━━━━━━━━━━━━━━━━━━━━ 4s 38ms/step - accuracy: 0.6491 - loss: 0.6244 - val_accuracy: 0.6239 - val_loss: 0.6585
Epoch 6/20
98/98 ━━━━━━━━━━━━━━━━━━━━ 5s 46ms/step - accuracy: 0.7022 - loss: 0.5794 - val_accuracy: 0.6138 - val_loss: 0.6874
Epoch 7/20
98/98 ━━━━━━━━━━━━━━━━━━━━ 4s 43ms/step - accuracy: 0.7617 - loss: 0.5018 - val_accuracy: 0.5994 - val_loss: 0.7765
Epoch 8/20
98/98 ━━━━━━━━━━━━━━━━━━━━ 4s 38ms/step - accuracy: 0.8189 - loss: 0.4033 - val_accuracy: 0.6037 - 

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


98/98 ━━━━━━━━━━━━━━━━━━━━ 8s 54ms/step - accuracy: 0.5186 - loss: 0.7112 - val_accuracy: 0.2032 - val_loss: 0.7611
Epoch 2/20
98/98 ━━━━━━━━━━━━━━━━━━━━ 4s 38ms/step - accuracy: 0.5136 - loss: 0.6907 - val_accuracy: 0.4856 - val_loss: 0.6989
Epoch 3/20
98/98 ━━━━━━━━━━━━━━━━━━━━ 4s 38ms/step - accuracy: 0.5321 - loss: 0.6789 - val_accuracy: 0.6225 - val_loss: 0.6640
Epoch 4/20
98/98 ━━━━━━━━━━━━━━━━━━━━ 5s 50ms/step - accuracy: 0.5468 - loss: 0.6855 - val_accuracy: 0.5259 - val_loss: 0.6909
Epoch 5/20
98/98 ━━━━━━━━━━━━━━━━━━━━ 4s 42ms/step - accuracy: 0.6222 - loss: 0.6456 - val_accuracy: 0.5259 - val_loss: 0.7022
Epoch 6/20
98/98 ━━━━━━━━━━━━━━━━━━━━ 4s 38ms/step - accuracy: 0.7048 - loss: 0.5759 - val_accuracy: 0.5548 - val_loss: 0.7039
55/55 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step
[KẾT QUẢ] Độ chính xác (Accuracy) cho trục NS: 59.31%
Báo cáo phân loại chi tiết (Classification Report):
              precision    recall  f1-score   support

           0     0.1480    0.4083    0.2173     

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


98/98 ━━━━━━━━━━━━━━━━━━━━ 8s 44ms/step - accuracy: 0.5089 - loss: 0.6994 - val_accuracy: 0.4582 - val_loss: 0.7126
Epoch 2/20
98/98 ━━━━━━━━━━━━━━━━━━━━ 4s 41ms/step - accuracy: 0.5529 - loss: 0.6855 - val_accuracy: 0.5576 - val_loss: 0.6866
Epoch 3/20
98/98 ━━━━━━━━━━━━━━━━━━━━ 5s 49ms/step - accuracy: 0.6020 - loss: 0.6688 - val_accuracy: 0.5720 - val_loss: 0.6844
Epoch 4/20
98/98 ━━━━━━━━━━━━━━━━━━━━ 4s 42ms/step - accuracy: 0.6250 - loss: 0.6502 - val_accuracy: 0.5461 - val_loss: 0.7064
Epoch 5/20
98/98 ━━━━━━━━━━━━━━━━━━━━ 4s 38ms/step - accuracy: 0.6339 - loss: 0.6288 - val_accuracy: 0.5634 - val_loss: 0.6920
Epoch 6/20
98/98 ━━━━━━━━━━━━━━━━━━━━ 4s 40ms/step - accuracy: 0.6961 - loss: 0.5804 - val_accuracy: 0.5937 - val_loss: 0.7116
55/55 ━━━━━━━━━━━━━━━━━━━━ 2s 24ms/step
[KẾT QUẢ] Độ chính xác (Accuracy) cho trục TF: 55.16%
Báo cáo phân loại chi tiết (Classification Report):
              precision    recall  f1-score   support

           0     0.5888    0.5687    0.5785     

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


98/98 ━━━━━━━━━━━━━━━━━━━━ 7s 44ms/step - accuracy: 0.4837 - loss: 0.7055 - val_accuracy: 0.4611 - val_loss: 0.7013
Epoch 2/20
98/98 ━━━━━━━━━━━━━━━━━━━━ 4s 43ms/step - accuracy: 0.5362 - loss: 0.6899 - val_accuracy: 0.5317 - val_loss: 0.6923
Epoch 3/20
98/98 ━━━━━━━━━━━━━━━━━━━━ 5s 47ms/step - accuracy: 0.5765 - loss: 0.6780 - val_accuracy: 0.5692 - val_loss: 0.6786
Epoch 4/20
98/98 ━━━━━━━━━━━━━━━━━━━━ 4s 39ms/step - accuracy: 0.5747 - loss: 0.6741 - val_accuracy: 0.5720 - val_loss: 0.6772
Epoch 5/20
98/98 ━━━━━━━━━━━━━━━━━━━━ 4s 39ms/step - accuracy: 0.6270 - loss: 0.6526 - val_accuracy: 0.6009 - val_loss: 0.6715
Epoch 6/20
98/98 ━━━━━━━━━━━━━━━━━━━━ 5s 50ms/step - accuracy: 0.6678 - loss: 0.6198 - val_accuracy: 0.5447 - val_loss: 0.7014
Epoch 7/20
98/98 ━━━━━━━━━━━━━━━━━━━━ 4s 38ms/step - accuracy: 0.7315 - loss: 0.5448 - val_accuracy: 0.5360 - val_loss: 0.7135
Epoch 8/20
98/98 ━━━━━━━━━━━━━━━━━━━━ 5s 39ms/step - accuracy: 0.7851 - loss: 0.4689 - val_accuracy: 0.5692 - val_loss: 0.

Thử GLOVE 300D + OVERSAMPLING

B1: Tải GloVe (Lần này dùng file 300d)

In [ ]:
import pandas as pd
import numpy as np
import re
import os
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, SpatialDropout1D, Bidirectional
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
from sklearn.utils import resample # Dùng để Oversampling

# Tải lại nếu chưa có (file zip chứa cả 300d)
if not os.path.exists('glove.6B.zip'):
    print("[THÔNG BÁO] Đang tải GloVe...")
    !wget http://nlp.stanford.edu/data/glove.6B.zip
    !unzip -q glove.6B.zip
else:
    print("[THÔNG BÁO] File GloVe đã có sẵn.")

print("[THÔNG BÁO] Đã sẵn sàng sử dụng GloVe 300d.")

[THÔNG BÁO] File GloVe đã có sẵn.
[THÔNG BÁO] Đã sẵn sàng sử dụng GloVe 300d.


B2: Tiền xử lý + Tokenize (Giữ nguyên)

In [ ]:
MAX_NB_WORDS = 20000
MAX_SEQUENCE_LENGTH = 500
EMBEDDING_DIM = 300 # <--- NÂNG CẤP LÊN 300 CHIỀU

# Load và Clean dữ liệu (Code cũ)
try:
    df = pd.read_csv('mbti_1.csv', on_bad_lines='skip', engine='python')
except:
    print("Upload file mbti_1.csv đi homie!")

def clean_text_formal(text):
    text = str(text).lower()
    text = re.sub(r'http\S+', '', text)
    mbti_types = ['infj', 'entp', 'intp', 'intj', 'entj', 'enfj', 'infp', 'enfp',
                  'isfp', 'istp', 'isfj', 'istj', 'estp', 'esfp', 'estj', 'esfj']
    pattern = r'\b(?:' + '|'.join(mbti_types) + r')\b'
    text = re.sub(pattern, '', text)
    text = re.sub(r'[^a-z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['clean_posts'] = df['posts'].apply(clean_text_formal)

def get_binary_labels(mbti_type):
    return pd.Series({
        'IE': 1 if 'I' in mbti_type else 0,
        'NS': 1 if 'N' in mbti_type else 0,
        'TF': 1 if 'T' in mbti_type else 0,
        'JP': 1 if 'J' in mbti_type else 0
    })
binary_targets = df['type'].apply(get_binary_labels)
df = pd.concat([df, binary_targets], axis=1)

# Tokenizer (Làm 1 lần cho toàn bộ data text)
print("[THÔNG BÁO] Đang Tokenize...")
tokenizer = Tokenizer(num_words=MAX_NB_WORDS, oov_token='<OOV>')
tokenizer.fit_on_texts(df['clean_posts'])

# Load GloVe 300d Matrix
print("[THÔNG BÁO] Đang nạp GloVe 300d...")
embeddings_index = {}
f = open('glove.6B.300d.txt', encoding='utf-8') # <--- Đọc file 300d
for line in f:
    values = line.split()
    word = values[0]
    coefs = np.asarray(values[1:], dtype='float32')
    embeddings_index[word] = coefs
f.close()

word_index = tokenizer.word_index
embedding_matrix = np.zeros((MAX_NB_WORDS, EMBEDDING_DIM))
hits = 0
for word, i in word_index.items():
    if i < MAX_NB_WORDS:
        embedding_vector = embeddings_index.get(word)
        if embedding_vector is not None:
            embedding_matrix[i] = embedding_vector
            hits += 1

print(f"[THỐNG KÊ] Số từ khớp GloVe 300d: {hits}")

[THÔNG BÁO] Đang Tokenize...
[THÔNG BÁO] Đang nạp GloVe 300d...
[THỐNG KÊ] Số từ khớp GloVe 300d: 19227


B3: Hàm Oversampling

In [ ]:
def oversample_data(X_text, y_label):
    """
    Hàm nhân bản dữ liệu nhóm thiểu số cho cân bằng 50/50
    """
    # Gộp lại thành DataFrame tạm để xử lý
    train_data = pd.DataFrame({'text': X_text, 'label': y_label})

    # Đếm số lượng
    count_0 = len(train_data[train_data['label']==0])
    count_1 = len(train_data[train_data['label']==1])

    if count_0 == count_1:
        return X_text, y_label # Đã cân bằng thì thôi

    # Xác định nhóm đa số và thiểu số
    if count_0 > count_1:
        df_majority = train_data[train_data['label']==0]
        df_minority = train_data[train_data['label']==1]
    else:
        df_majority = train_data[train_data['label']==1]
        df_minority = train_data[train_data['label']==0]

    # Oversample nhóm thiểu số
    df_minority_upsampled = resample(df_minority,
                                     replace=True,     # Cho phép chọn lại (nhân bản)
                                     n_samples=len(df_majority), # Cho bằng nhóm đa số
                                     random_state=42)

    # Gộp lại
    df_upsampled = pd.concat([df_majority, df_minority_upsampled])

    # Trộn đều (Shuffle)
    df_upsampled = df_upsampled.sample(frac=1, random_state=42).reset_index(drop=True)

    return df_upsampled['text'], df_upsampled['label']

B4: Train với Oversampling

In [ ]:
EPOCHS = 20
BATCH_SIZE = 64

def build_model_300d():
    model = Sequential()
    model.add(Embedding(MAX_NB_WORDS, EMBEDDING_DIM,
                        input_length=MAX_SEQUENCE_LENGTH,
                        weights=[embedding_matrix],
                        trainable=True)) # Vẫn cho Fine-tune
    model.add(SpatialDropout1D(0.3)) # Tăng dropout lên 0.3 vì dùng 300d dễ overfitting
    model.add(Bidirectional(LSTM(128, dropout=0.3, recurrent_dropout=0.0))) # Tăng unit lên 128
    model.add(Dense(1, activation='sigmoid'))
    model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
    return model

target_cols = ['IE', 'NS', 'TF', 'JP']

print("="*60)
print("🚀 CHIẾN DỊCH TẤT TAY: GLOVE 300D + OVERSAMPLING")
print("="*60)

for col in target_cols:
    print(f"\n[THÔNG BÁO] Đang xử lý trục: {col}...")

    # 1. Chia Train/Test (Split trước, Oversample sau -> CHUẨN KHOA HỌC)
    X_train_txt, X_test_txt, y_train, y_test = train_test_split(
        df['clean_posts'], df[col], test_size=0.2, random_state=42, stratify=df['type']
    )

    # 2. Oversampling (Chỉ làm trên tập Train)
    print(f"   -> Trước khi cân bằng: {y_train.value_counts().to_dict()}")
    X_train_resampled, y_train_resampled = oversample_data(X_train_txt, y_train)
    print(f"   -> Sau khi cân bằng: {y_train_resampled.value_counts().to_dict()}")

    # 3. Tokenize sang dạng số
    X_train_seq = tokenizer.texts_to_sequences(X_train_resampled)
    X_test_seq = tokenizer.texts_to_sequences(X_test_txt)

    X_train_pad = pad_sequences(X_train_seq, maxlen=MAX_SEQUENCE_LENGTH)
    X_test_pad = pad_sequences(X_test_seq, maxlen=MAX_SEQUENCE_LENGTH)

    # 4. Train
    model = build_model_300d()
    early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

    # Không cần class_weight nữa vì dữ liệu đã cân bằng 50/50
    model.fit(X_train_pad, y_train_resampled,
              epochs=EPOCHS, batch_size=BATCH_SIZE,
              validation_split=0.1,
              callbacks=[early_stop],
              verbose=1)

    # 5. Đánh giá
    y_pred = (model.predict(X_test_pad) > 0.5).astype(int)
    print(f"[KẾT QUẢ] Accuracy trục {col}: {accuracy_score(y_test, y_pred):.2%}")
    print(classification_report(y_test, y_pred, digits=4))

print("\n✅ DONE! Hy vọng kết quả sẽ rực rỡ hơn!")

🚀 CHIẾN DỊCH TẤT TAY: GLOVE 300D + OVERSAMPLING

[THÔNG BÁO] Đang xử lý trục: IE...
   -> Trước khi cân bằng: {1: 5342, 0: 1598}
   -> Sau khi cân bằng: {0: 5342, 1: 5342}
Epoch 1/20


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


151/151 ━━━━━━━━━━━━━━━━━━━━ 15s 75ms/step - accuracy: 0.5248 - loss: 0.6958 - val_accuracy: 0.6090 - val_loss: 0.6556
Epoch 2/20
151/151 ━━━━━━━━━━━━━━━━━━━━ 11s 74ms/step - accuracy: 0.6624 - loss: 0.6228 - val_accuracy: 0.7007 - val_loss: 0.5670
Epoch 3/20
151/151 ━━━━━━━━━━━━━━━━━━━━ 11s 74ms/step - accuracy: 0.7790 - loss: 0.4754 - val_accuracy: 0.7933 - val_loss: 0.4338
Epoch 4/20
151/151 ━━━━━━━━━━━━━━━━━━━━ 11s 74ms/step - accuracy: 0.8778 - loss: 0.2975 - val_accuracy: 0.8457 - val_loss: 0.3669
Epoch 5/20
151/151 ━━━━━━━━━━━━━━━━━━━━ 11s 73ms/step - accuracy: 0.9327 - loss: 0.1756 - val_accuracy: 0.8700 - val_loss: 0.3825
Epoch 6/20
151/151 ━━━━━━━━━━━━━━━━━━━━ 11s 73ms/step - accuracy: 0.9618 - loss: 0.1094 - val_accuracy: 0.9065 - val_loss: 0.3075
Epoch 7/20
151/151 ━━━━━━━━━━━━━━━━━━━━ 11s 74ms/step - accuracy: 0.9751 - loss: 0.0740 - val_accuracy: 0.8831 - val_loss: 0.4100
Epoch 8/20
151/151 ━━━━━━━━━━━━━━━━━━━━ 11s 74ms/step - accuracy: 0.9858 - loss: 0.0444 - val_accurac

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Epoch 1/20
169/169 ━━━━━━━━━━━━━━━━━━━━ 16s 81ms/step - accuracy: 0.5355 - loss: 0.6936 - val_accuracy: 0.6976 - val_loss: 0.6113
Epoch 2/20
169/169 ━━━━━━━━━━━━━━━━━━━━ 13s 74ms/step - accuracy: 0.7182 - loss: 0.5581 - val_accuracy: 0.8454 - val_loss: 0.3698
Epoch 3/20
169/169 ━━━━━━━━━━━━━━━━━━━━ 13s 74ms/step - accuracy: 0.8764 - loss: 0.2899 - val_accuracy: 0.9039 - val_loss: 0.2418
Epoch 4/20
169/169 ━━━━━━━━━━━━━━━━━━━━ 12s 74ms/step - accuracy: 0.9433 - loss: 0.1558 - val_accuracy: 0.9415 - val_loss: 0.1704
Epoch 5/20
169/169 ━━━━━━━━━━━━━━━━━━━━ 12s 74ms/step - accuracy: 0.9771 - loss: 0.0704 - val_accuracy: 0.9340 - val_loss: 0.1954
Epoch 6/20
169/169 ━━━━━━━━━━━━━━━━━━━━ 21s 74ms/step - accuracy: 0.9815 - loss: 0.0509 - val_accuracy: 0.9457 - val_loss: 0.1732
Epoch 7/20
169/169 ━━━━━━━━━━━━━━━━━━━━ 13s 74ms/step - accuracy: 0.9896 - loss: 0.0288 - val_accuracy: 0.9524 - val_loss: 0.1451
Epoch 8/20
169/169 ━━━━━━━━━━━━━━━━━━━━ 13s 74ms/step - accuracy: 0.9917 - loss: 0.0249 - 

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


106/106 ━━━━━━━━━━━━━━━━━━━━ 11s 79ms/step - accuracy: 0.5159 - loss: 0.7014 - val_accuracy: 0.5979 - val_loss: 0.6687
Epoch 2/20
106/106 ━━━━━━━━━━━━━━━━━━━━ 8s 76ms/step - accuracy: 0.6193 - loss: 0.6560 - val_accuracy: 0.6019 - val_loss: 0.6508
Epoch 3/20
106/106 ━━━━━━━━━━━━━━━━━━━━ 8s 74ms/step - accuracy: 0.6802 - loss: 0.5948 - val_accuracy: 0.6591 - val_loss: 0.6107
Epoch 4/20
106/106 ━━━━━━━━━━━━━━━━━━━━ 8s 78ms/step - accuracy: 0.7551 - loss: 0.4963 - val_accuracy: 0.7217 - val_loss: 0.5751
Epoch 5/20
106/106 ━━━━━━━━━━━━━━━━━━━━ 8s 73ms/step - accuracy: 0.8608 - loss: 0.3378 - val_accuracy: 0.7537 - val_loss: 0.5312
Epoch 6/20
106/106 ━━━━━━━━━━━━━━━━━━━━ 8s 76ms/step - accuracy: 0.9075 - loss: 0.2309 - val_accuracy: 0.7297 - val_loss: 0.6131
Epoch 7/20
106/106 ━━━━━━━━━━━━━━━━━━━━ 10s 73ms/step - accuracy: 0.9498 - loss: 0.1358 - val_accuracy: 0.8003 - val_loss: 0.5569
Epoch 8/20
106/106 ━━━━━━━━━━━━━━━━━━━━ 8s 74ms/step - accuracy: 0.9707 - loss: 0.0839 - val_accuracy: 0.7

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


118/118 ━━━━━━━━━━━━━━━━━━━━ 12s 75ms/step - accuracy: 0.5159 - loss: 0.7059 - val_accuracy: 0.5745 - val_loss: 0.6826
Epoch 2/20
118/118 ━━━━━━━━━━━━━━━━━━━━ 9s 75ms/step - accuracy: 0.5891 - loss: 0.6685 - val_accuracy: 0.6150 - val_loss: 0.6615
Epoch 3/20
118/118 ━━━━━━━━━━━━━━━━━━━━ 9s 77ms/step - accuracy: 0.6786 - loss: 0.6016 - val_accuracy: 0.6663 - val_loss: 0.6222
Epoch 4/20
118/118 ━━━━━━━━━━━━━━━━━━━━ 9s 74ms/step - accuracy: 0.7699 - loss: 0.4830 - val_accuracy: 0.6722 - val_loss: 0.6069
Epoch 5/20
118/118 ━━━━━━━━━━━━━━━━━━━━ 9s 76ms/step - accuracy: 0.8515 - loss: 0.3481 - val_accuracy: 0.7259 - val_loss: 0.5701
Epoch 6/20
118/118 ━━━━━━━━━━━━━━━━━━━━ 9s 76ms/step - accuracy: 0.9038 - loss: 0.2379 - val_accuracy: 0.7342 - val_loss: 0.7212
Epoch 7/20
118/118 ━━━━━━━━━━━━━━━━━━━━ 9s 74ms/step - accuracy: 0.9388 - loss: 0.1557 - val_accuracy: 0.7473 - val_loss: 0.7398
Epoch 8/20
118/118 ━━━━━━━━━━━━━━━━━━━━ 9s 75ms/step - accuracy: 0.9646 - loss: 0.1069 - val_accuracy: 0.74

THỬ NGHIỆM KĨ THUẬT FAST TEXT, FOCAL LOSS VÀ THÊM LỚP ATTENTION CHO LSTM

FAST TEST + FOCAL LOSS + LSTM(ATTENTION)

 B1: Định nghĩa lớp Attention + hàm Focal Loss

In [6]:
import tensorflow as tf
from tensorflow.keras import backend as K
from tensorflow.keras.layers import Layer, Input, Embedding, LSTM, Dense, SpatialDropout1D, Bidirectional, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping
import numpy as np
import pandas as pd
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score

# 1. ĐỊNH NGHĨA FOCAL LOSS (Để trị mất cân bằng dữ liệu)
def focal_loss(gamma=2., alpha=0.25):
    def focal_loss_fixed(y_true, y_pred):
        pt_1 = tf.where(tf.equal(y_true, 1), y_pred, tf.ones_like(y_pred))
        pt_0 = tf.where(tf.equal(y_true, 0), y_pred, tf.zeros_like(y_pred))
        return -K.sum(alpha * K.pow(1. - pt_1, gamma) * K.log(pt_1)) - K.sum((1 - alpha) * K.pow(pt_0, gamma) * K.log(1. - pt_0))
    return focal_loss_fixed

# 2. ĐỊNH NGHĨA LAYER ATTENTION (Để LSTM nhớ được ngữ cảnh quan trọng)
class Attention(Layer):
    def __init__(self, step_dim, W_regularizer=None, b_regularizer=None, W_constraint=None, b_constraint=None, bias=True, **kwargs):
        self.supports_masking = True
        self.init = tf.keras.initializers.get('glorot_uniform')
        self.W_regularizer = tf.keras.regularizers.get(W_regularizer)
        self.b_regularizer = tf.keras.regularizers.get(b_regularizer)
        self.W_constraint = tf.keras.constraints.get(W_constraint)
        self.b_constraint = tf.keras.constraints.get(b_constraint)
        self.bias = bias
        self.step_dim = step_dim
        self.features_dim = 0
        super(Attention, self).__init__(**kwargs)

    def build(self, input_shape):
        assert len(input_shape) == 3
        self.W = self.add_weight(shape=(input_shape[-1],),
                                 initializer=self.init,
                                 name='{}_W'.format(self.name),
                                 regularizer=self.W_regularizer,
                                 constraint=self.W_constraint)
        self.features_dim = input_shape[-1]
        if self.bias:
            self.b = self.add_weight(shape=(input_shape[1],),
                                     initializer='zero',
                                     name='{}_b'.format(self.name),
                                     regularizer=self.b_regularizer,
                                     constraint=self.b_constraint)
        else:
            self.b = None
        self.built = True

    def compute_mask(self, input, input_mask=None):
        return None

    def call(self, x, mask=None):
        features_dim = self.features_dim
        step_dim = self.step_dim
        eij = K.reshape(K.dot(K.reshape(x, (-1, features_dim)), K.reshape(self.W, (features_dim, 1))), (-1, step_dim))
        if self.bias: eij += self.b
        eij = K.tanh(eij)
        a = K.exp(eij)
        if mask is not None: a *= K.cast(mask, K.floatx())
        a /= K.cast(K.sum(a, axis=1, keepdims=True) + K.epsilon(), K.floatx())
        a = K.expand_dims(a)
        weighted_input = x * a
        return K.sum(weighted_input, axis=1)

    def get_config(self):
        return super(Attention, self).get_config()

print("[BƯỚC 1] Đã khởi tạo xong Focal Loss và Attention Layer!")

[BƯỚC 1] Đã khởi tạo xong Focal Loss và Attention Layer!


Tải FastText

In [2]:
import os

print("[INFO] Khởi tạo quy trình tải xuống bộ dữ liệu FastText (Wiki News 300d)...")
print("[INFO] Quá trình tải xuống có thể mất vài phút tùy thuộc vào băng thông mạng.")

# 1. Tải file zip từ kho lưu trữ công khai
!wget -N https://dl.fbaipublicfiles.com/fasttext/vectors-english/wiki-news-300d-1M.vec.zip

# 2. Giải nén tập tin
print("[INFO] Đang giải nén tập tin dữ liệu...")
!unzip -n -q wiki-news-300d-1M.vec.zip

print("[SUCCESS] Đã tải và giải nén thành công 'wiki-news-300d-1M.vec'. Sẵn sàng cho bước tiếp theo.")

[INFO] Khởi tạo quy trình tải xuống bộ dữ liệu FastText (Wiki News 300d)...
[INFO] Quá trình tải xuống có thể mất vài phút tùy thuộc vào băng thông mạng.
--2025-12-19 07:54:04--  https://dl.fbaipublicfiles.com/fasttext/vectors-english/wiki-news-300d-1M.vec.zip
Resolving dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)... 65.8.76.89, 65.8.76.35, 65.8.76.47, ...
Connecting to dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)|65.8.76.89|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 681808098 (650M) [application/zip]
Saving to: ‘wiki-news-300d-1M.vec.zip’

wiki-news-300d-1M.v 100%[===================>] 650.22M  22.8MB/s    in 29s     

2025-12-19 07:54:34 (22.1 MB/s) - ‘wiki-news-300d-1M.vec.zip’ saved [681808098/681808098]

[INFO] Đang giải nén tập tin dữ liệu...
[SUCCESS] Đã tải và giải nén thành công 'wiki-news-300d-1M.vec'. Sẵn sàng cho bước tiếp theo.


B2: TIỀN XỬ LÝ & VECTOR HÓA

In [7]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
import re

# Cấu hình siêu tham số (Hyperparameters configuration)
MAX_NB_WORDS = 20000       # Số lượng từ vựng tối đa
MAX_SEQUENCE_LENGTH = 500  # Độ dài chuỗi đầu vào
EMBEDDING_DIM = 300        # Kích thước vector FastText

# 1. Tải và Làm sạch dữ liệu
print("[INFO] Đang tải tập dữ liệu MBTI...")
df = pd.read_csv('mbti_1.csv', on_bad_lines='skip', engine='python')

def clean_text_formal(text):
    """
    Hàm chuẩn hóa văn bản: Loại bỏ URL, ký tự đặc biệt và các nhãn MBTI (để tránh rò rỉ dữ liệu).
    """
    text = str(text).lower()
    text = re.sub(r'http\S+', '', text)
    mbti_types = ['infj', 'entp', 'intp', 'intj', 'entj', 'enfj', 'infp', 'enfp', 'isfp', 'istp', 'isfj', 'istj', 'estp', 'esfp', 'estj', 'esfj']
    pattern = r'\b(?:' + '|'.join(mbti_types) + r')\b'
    text = re.sub(pattern, '', text)
    text = re.sub(r'[^a-z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['clean_posts'] = df['posts'].apply(clean_text_formal)

# Mã hóa nhãn (Label Encoding)
def get_binary_labels(mbti_type):
    return pd.Series({'IE': 1 if 'I' in mbti_type else 0, 'NS': 1 if 'N' in mbti_type else 0, 'TF': 1 if 'T' in mbti_type else 0, 'JP': 1 if 'J' in mbti_type else 0})
df = pd.concat([df, df['type'].apply(get_binary_labels)], axis=1)

# 2. Tokenization
print("[INFO] Đang thực hiện mã hóa văn bản (Tokenization)...")
tokenizer = Tokenizer(num_words=MAX_NB_WORDS, oov_token='<OOV>')
tokenizer.fit_on_texts(df['clean_posts'])
word_index = tokenizer.word_index

# 3. Tạo Ma trận Embedding (Embedding Matrix Construction)
print("[INFO] Đang nạp vector từ tập tin FastText...")
embeddings_index = {}
with open('wiki-news-300d-1M.vec', encoding='utf-8', errors='ignore') as f:
    for i, line in enumerate(f):
        if i == 0: continue
        values = line.split()
        word = values[0]
        try: embeddings_index[word] = np.asarray(values[1:], dtype='float32')
        except: continue

print("[INFO] Đang xây dựng ma trận trọng số Embedding...")
embedding_matrix = np.zeros((MAX_NB_WORDS, EMBEDDING_DIM))
hits = 0
for word, i in word_index.items():
    if i < MAX_NB_WORDS:
        embedding_vector = embeddings_index.get(word)
        if embedding_vector is not None:
            embedding_matrix[i] = embedding_vector
            hits += 1

print(f"[SUCCESS] Hoàn tất tiền xử lý. Số lượng từ vựng khớp với FastText: {hits}")

[INFO] Đang tải tập dữ liệu MBTI...
[INFO] Đang thực hiện mã hóa văn bản (Tokenization)...
[INFO] Đang nạp vector từ tập tin FastText...
[INFO] Đang xây dựng ma trận trọng số Embedding...
[SUCCESS] Hoàn tất tiền xử lý. Số lượng từ vựng khớp với FastText: 19318


B3: XÂY DỰNG KIẾN TRÚC MÔ HÌNH

In [8]:
from tensorflow.keras.layers import Input, Embedding, LSTM, Dense, SpatialDropout1D, Bidirectional, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam

def build_attention_lstm_model():
    inputs = Input(shape=(MAX_SEQUENCE_LENGTH,))

    # 1. FastText Embedding
    x = Embedding(MAX_NB_WORDS, EMBEDDING_DIM,
                  weights=[embedding_matrix],
                  trainable=True)(inputs)
    x = SpatialDropout1D(0.3)(x)

    # 2. Bi-Directional LSTM (128 units - Cấu hình mạnh)
    # return_sequences=True để Attention hoạt động
    x = Bidirectional(LSTM(128, return_sequences=True, dropout=0.3, recurrent_dropout=0.0))(x)

    # 3. Attention Layer
    x = Attention(MAX_SEQUENCE_LENGTH)(x)

    # 4. Dense Layer (64 units)
    x = Dense(64, activation='relu')(x)
    x = Dropout(0.3)(x)

    # Output
    outputs = Dense(1, activation='sigmoid')(x)

    model = Model(inputs=inputs, outputs=outputs)

    # FOCAL LOSS + ADAM (Không clipnorm, để nó học tự nhiên)
    model.compile(loss=focal_loss(gamma=2., alpha=0.25),
                  optimizer='adam',
                  metrics=['accuracy'])
    return model

print("[INFO] Đã khôi phục kiến trúc LSTM + Attention mạnh nhất!")

[INFO] Đã khôi phục kiến trúc LSTM + Attention mạnh nhất!


B4: HUẤN LUYỆN & ĐÁNH GIÁ

In [9]:
target_cols = ['IE', 'NS', 'TF', 'JP']
EPOCHS = 20
BATCH_SIZE = 64

print("="*60)
print("🚀 KHÔI PHỤC CHIẾN DỊCH: LSTM + ATTENTION + FOCAL LOSS (BEST VERSION)")
print("="*60)

for col in target_cols:
    print(f"\n[INFO] Đang xử lý trục: {col}...")

    # 1. Chia data
    X_train_txt, X_test_txt, y_train, y_test = train_test_split(
        df['clean_posts'], df[col], test_size=0.2, random_state=42, stratify=df['type']
    )

    # 2. Tokenize
    X_train_seq = tokenizer.texts_to_sequences(X_train_txt)
    X_test_seq = tokenizer.texts_to_sequences(X_test_txt)
    X_train_pad = pad_sequences(X_train_seq, maxlen=MAX_SEQUENCE_LENGTH)
    X_test_pad = pad_sequences(X_test_seq, maxlen=MAX_SEQUENCE_LENGTH)

    # 3. Gọi Model
    model = build_attention_lstm_model()

    # 4. Train (Chỉ dùng EarlyStopping)
    early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

    model.fit(X_train_pad, y_train,
              epochs=EPOCHS, batch_size=BATCH_SIZE,
              validation_split=0.1,
              callbacks=[early_stop],
              verbose=1)

    # 5. Đánh giá
    y_pred = (model.predict(X_test_pad) > 0.5).astype(int)
    print(f"[KẾT QUẢ] Accuracy trục {col}: {accuracy_score(y_test, y_pred):.2%}")
    print(classification_report(y_test, y_pred, digits=4))

print("\n✅ Đã lấy lại phong độ!")

🚀 KHÔI PHỤC CHIẾN DỊCH: LSTM + ATTENTION + FOCAL LOSS (BEST VERSION)

[INFO] Đang xử lý trục: IE...
Epoch 1/20
98/98 ━━━━━━━━━━━━━━━━━━━━ 13s 91ms/step - accuracy: 0.5854 - loss: 4.1284 - val_accuracy: 0.7536 - val_loss: 4.1264
Epoch 2/20
98/98 ━━━━━━━━━━━━━━━━━━━━ 8s 82ms/step - accuracy: 0.6741 - loss: 3.9668 - val_accuracy: 0.4524 - val_loss: 3.9753
Epoch 3/20
98/98 ━━━━━━━━━━━━━━━━━━━━ 8s 81ms/step - accuracy: 0.6566 - loss: 3.6420 - val_accuracy: 0.6873 - val_loss: 3.8366
Epoch 4/20
98/98 ━━━━━━━━━━━━━━━━━━━━ 8s 81ms/step - accuracy: 0.7929 - loss: 2.8698 - val_accuracy: 0.7435 - val_loss: 7.2996
Epoch 5/20
98/98 ━━━━━━━━━━━━━━━━━━━━ 8s 79ms/step - accuracy: 0.8921 - loss: 1.6346 - val_accuracy: 0.6787 - val_loss: 7.3659
Epoch 6/20
98/98 ━━━━━━━━━━━━━━━━━━━━ 8s 81ms/step - accuracy: 0.9575 - loss: 0.7375 - val_accuracy: 0.7305 - val_loss: 13.7835
55/55 ━━━━━━━━━━━━━━━━━━━━ 1s 21ms/step
[KẾT QUẢ] Accuracy trục IE: 72.05%
              precision    recall  f1-score   support

      